In [61]:
import pandas as pd
import numpy as np

In [62]:
import kagglehub
import os

path = kagglehub.dataset_download("xvniyusheng/mindsmall-train")

In [63]:
print(path)
print(os.listdir(path))

C:\Users\HONOR\.cache\kagglehub\datasets\xvniyusheng\mindsmall-train\versions\1
['behaviors.tsv', 'entity_embedding.vec', 'news.tsv', 'relation_embedding.vec']


In [64]:
from pathlib import Path

path = Path(path)

In [ ]:
# на странице датасета такие колонки

behaviors = pd.read_csv(
    path / "behaviors.tsv",
    sep="\t",
    header=None,
    names=['impression_id', 'user_id', 'time', 'history', 'impressions']
)

news = pd.read_csv(
    path / "news.tsv",
    sep="\t",
    header=None,
    names=[
                  'id', 'category', 'subcategory', 'title', 'abstract', 'url',
                  'title_entities', 'abstract_entities'
              ]
)

print(behaviors.shape)
print(news.shape)

(156965, 5)
(51282, 8)


In [66]:
behaviors.head(5)

,impression_id,user_id,time,history,impressions
0,1,U13740,11/11/2019 9:05:58 AM,N55189 N42782 N34694 N45794 N18445 N63302 N104...,N55689-1 N35729-0
1,2,U91836,11/12/2019 6:11:30 PM,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,N20678-0 N39317-0 N58114-0 N20495-0 N42977-0 N...
2,3,U73700,11/14/2019 7:01:48 AM,N10732 N25792 N7563 N21087 N41087 N5445 N60384...,N50014-0 N23877-0 N35389-0 N49712-0 N16844-0 N...
3,4,U34670,11/11/2019 5:28:05 AM,N45729 N2203 N871 N53880 N41375 N43142 N33013 ...,N35729-0 N33632-0 N49685-1 N27581-0
4,5,U8125,11/12/2019 4:11:21 PM,N10078 N56514 N14904 N33740,N39985-0 N36050-0 N16096-0 N8400-1 N22407-0 N6...


In [67]:
news.head(5)

,id,category,subcategory,title,abstract,url,title_entities,abstract_entities
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the...",https://assets.msn.com/labs/mind/AAGH0ET.html,"[{""Label"": ""Prince Philip, Duke of Edinburgh"",...",[]
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...,https://assets.msn.com/labs/mind/AAB19MK.html,"[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik...","[{""Label"": ""Adipose tissue"", ""Type"": ""C"", ""Wik..."
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...,https://assets.msn.com/labs/mind/AAJgNsz.html,[],"[{""Label"": ""Ukraine"", ""Type"": ""G"", ""WikidataId..."
3,N53526,health,voices,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi...",https://assets.msn.com/labs/mind/AACk2N6.html,[],"[{""Label"": ""National Basketball Association"", ..."
4,N38324,health,medical,"How to Get Rid of Skin Tags, According to a De...","They seem harmless, but there's a very good re...",https://assets.msn.com/labs/mind/AAAKEkt.html,"[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI...","[{""Label"": ""Skin tag"", ""Type"": ""C"", ""WikidataI..."


In [68]:
behaviors = behaviors.dropna(subset=["history"]).copy()

behaviors["history"] = behaviors["history"].apply(
    lambda x: x.split() if isinstance(x, str) else []
)

behaviors = behaviors[behaviors["history"].map(len) > 0]

interactions = behaviors[["user_id", "history"]].explode("history")

interactions = interactions.rename(
    columns={"history": "item_id"}
)

interactions["timestamp"] = (
    interactions.groupby("user_id").cumcount()
)

interactions = interactions[
    ["user_id", "item_id", "timestamp"]
].reset_index(drop=True)

interactions.head()

,user_id,item_id,timestamp
0,U13740,N55189,0
1,U13740,N42782,1
2,U13740,N34694,2
3,U13740,N45794,3
4,U13740,N18445,4


In [69]:
print("Interactions:", interactions.shape)
print("Users:", interactions["user_id"].nunique())
print("Items:", interactions["item_id"].nunique())

print(interactions.groupby("user_id").size().describe())

Interactions: (5107639, 3)
Users: 49108
Items: 33195
count    49108.000000
mean       104.008288
std        349.703277
min          1.000000
25%          8.000000
50%         20.000000
75%         70.000000
max      18972.000000
dtype: float64


In [70]:
from replay.preprocessing import LabelEncoder
from replay.preprocessing import LabelEncodingRule

encoder = LabelEncoder(
    [
        LabelEncodingRule("user_id"),
        LabelEncodingRule("item_id"),
    ]
)

interactions_encoded = encoder.fit_transform(interactions)

interactions_encoded.head()

,timestamp,user_id,item_id
0,0,2201,25615
1,1,2201,18583
2,2,2201,14002
3,3,2201,20321
4,4,2201,4775


In [74]:
from replay.splitters import LastNSplitter

splitter = LastNSplitter(
    N=1,
    divide_column="user_id",
    query_column="user_id",
    strategy="interactions",
    drop_cold_users=True,
    drop_cold_items=True
)

train, validation = splitter.split(interactions_encoded)


In [77]:
train.head()

,timestamp,user_id,item_id
3517976,0,0,5779
3517977,1,0,13606
3517978,2,0,20415
3517979,3,0,25922
3517980,4,0,22742


In [73]:
print(train.shape)
print(validation.shape)

print(train.head())
print(validation.head())

(5058531, 3)
(48769, 3)
        user_id item_id  timestamp
3517976    U100  N20121          0
3517977    U100  N33998          1
3517978    U100  N45954          2
3517979    U100  N55743          3
3517980    U100  N50095          4
        user_id item_id  timestamp
3517985    U100  N20575          9
2831885   U1000  N41244          8
5074961  U10001  N59380         29
4774843  U10003  N28257         15
1528294  U10008   N5812         22


### пока не уверен в чистоте тех данных, тестил команды в основном